## 1. Importing libraries

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import zscore, skew
import seaborn as sns
import matplotlib.pyplot as plt
import sys
sys.path.append("../scripts")
from outlier_utils import print_outliers

## 2. Data Loading 

In [3]:
df = pd.read_csv("../data/nigeria.csv")
df.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M
0,2015,1,25.23,29.25,22.06,7.19,0.0,68.26,1.73,2.61,100.86,13.36
1,2015,2,26.16,29.41,22.87,6.54,0.0,73.23,1.42,1.95,100.94,15.37
2,2015,3,25.66,29.02,22.63,6.39,0.0,78.71,1.69,2.33,101.06,15.98
3,2015,4,24.11,27.27,19.92,7.35,0.0,63.66,2.15,3.80,101.09,11.65
4,2015,5,23.40,27.28,18.18,9.10,0.0,59.45,1.88,3.48,101.03,10.40


## 3. Adding column & Date Parsing

In [4]:
# Add country column
df["Country"] = "Nigeria"

# Convert YEAR + DOY to datetime
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract month
df["Month"] = df["Date"].dt.month

df.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,Month
0,2015,1,25.23,29.25,22.06,7.19,0.0,68.26,1.73,2.61,100.86,13.36,Nigeria,2015-01-01,1
1,2015,2,26.16,29.41,22.87,6.54,0.0,73.23,1.42,1.95,100.94,15.37,Nigeria,2015-01-02,1
2,2015,3,25.66,29.02,22.63,6.39,0.0,78.71,1.69,2.33,101.06,15.98,Nigeria,2015-01-03,1
3,2015,4,24.11,27.27,19.92,7.35,0.0,63.66,2.15,3.80,101.09,11.65,Nigeria,2015-01-04,1
4,2015,5,23.40,27.28,18.18,9.10,0.0,59.45,1.88,3.48,101.03,10.40,Nigeria,2015-01-05,1


- The dataset encodes dates using Year and Day-of-Year (DOY), which was converted into a proper datetime format to enable time-series analysis and seasonal interpretations.

## 4. Summary Statistics & Missing Values

In [5]:
# Replace sentinel values
df.replace(-999, np.nan, inplace=True)

# Duplicates
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

df = df.drop_duplicates()

Duplicate rows: 0


- No duplicates were found in any of the columns, which means every entry is unique

In [6]:
# Summary statistics
df.describe()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Date,Month
count,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108,4108.000000
mean,2020.131451,180.121227,26.656928,28.914667,24.886461,4.028206,4.213914,85.237040,2.217135,2.903335,100.827205,18.558505,2020-08-15 12:00:00,6.423564
min,2015.000000,1.000000,21.120000,25.260000,15.170000,1.160000,0.000000,54.400000,0.740000,1.290000,100.380000,9.430000,2015-01-01 00:00:00,1.000000
25%,2017.000000,86.000000,25.720000,27.920000,24.100000,3.090000,0.330000,83.930000,1.770000,2.370000,100.710000,17.970000,2017-10-23 18:00:00,3.000000
50%,2020.000000,179.000000,26.820000,28.990000,25.100000,3.770000,1.840000,86.350000,2.200000,2.810000,100.820000,18.840000,2020-08-15 12:00:00,6.000000
75%,2023.000000,272.000000,27.540000,29.910000,25.860000,4.600000,5.200000,88.500000,2.630000,3.390000,100.950000,19.570000,2023-06-08 06:00:00,9.000000
max,2026.000000,366.000000,29.290000,32.880000,27.790000,11.730000,166.100000,93.790000,4.780000,6.000000,101.350000,21.740000,2026-03-31 00:00:00,12.000000
std,3.248907,106.294767,1.123335,1.294345,1.396727,1.399169,7.266742,5.446007,0.587191,0.696885,0.165321,1.646313,NaN,3.477046


### Brief interpretation of the statistics
### **Overall Observation Count**
- **Count = 4108** for all variables (except a few with 4108.00x due to floating-point representation).  
  This means the dataset contains **4108 daily records**.

---

### **YEAR** (Year of observation)
- **Range**: 2015 – 2026 (approx. 12 years)  
---

### **T2M** (Mean daily air temperature at 2m, °C)
- **Mean**: 26.66°C → warm tropical/subtropical climate  
- **Min**: 21.12°C, **Max**: 29.29°C → relatively narrow range (no extreme cold)  
- **Std**: 1.12 → low variability → stable warm temperatures year-round

---

### **T2M_MAX** (Maximum daily temperature, °C)
- **Mean**: 28.91°C  
- **Min**: 25.26°C, **Max**: 32.88°C  
- Hot days up to ~33°C

---

### **T2M_MIN** (Minimum daily temperature, °C)
- **Mean**: 24.88°C  
- **Min**: 15.17°C (rare cool night), **Max**: 27.79°C  
- Nights are warm, rarely dropping below 15°C

---

### **T2M_RANGE** (Daily temperature range, °C)
- **Mean**: 4.03°C → small diurnal variation  
- **Min**: 1.16°C, **Max**: 11.73°C  
- Most days have a small range (typical of humid tropical climates)

---

### **PRECTOTCORR** (Bias-corrected total daily precipitation, mm/day)
- **Mean**: 4.21 mm/day  
- **Median**: 1.84 mm/day → distribution is right-skewed (many low-rain days, few heavy-rain days)  
- **Min**: 0 mm (dry days), **Max**: 166.1 mm → extreme heavy rain event  
- **Std**: 7.27 → high variability

---

### **RH2M** (Relative humidity at 2m, %)
- **Mean**: 85.24% → very humid  
- **Min**: 54.4%, **Max**: 93.79%  
- 25th percentile: 83.9%, 75th: 88.5% → consistently high humidity

---

### **WS2M** (Mean daily wind speed at 2m, m/s)
- **Mean**: 2.22 m/s (light breeze)  
- **Min**: 0.74 m/s, **Max**: 4.78 m/s  
- Generally calm to light winds

---

### **WS2M_MAX** (Maximum daily wind speed, m/s)
- **Mean**: 2.90 m/s  
- **Max**: 6.00 m/s → occasional moderate gusts

---

### **PS** (Atmospheric surface pressure, kPa)
- **Mean**: 100.83 kPa (≈ 1008.3 hPa) → near standard sea-level pressure  
- **Range**: 100.38 – 101.35 kPa → very stable pressure environment

---

### **QV2M** (Specific humidity, g/kg)
- **Mean**: 18.56 g/kg → very humid air (typical of tropics)  
- **Min**: 9.43 g/kg, **Max**: 21.74 g/kg  
- **Std**: 1.65 → some seasonal/dry-season variation

---

## **Key Climate Summary**
- **Climate type**: Warm, humid, tropical (likely coastal or lowland tropical rainforest/savanna)
- **Temperature**: Very stable year-round (~21–29°C), small daily range (~4°C)
- **Rainfall**: Highly variable – mostly light rain days with occasional extreme downpours (up to 166 mm/day)
- **Humidity**: Consistently high (85% average)
- **Wind**: Light, rarely strong
- **Pressure**: Very stable

This location experiences **no cold season**, **high moisture levels**, and **rainfall driven by convective storms** rather than frontal systems.

## 5. Missing values

In [7]:
# Missing values
missing = df.isna().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

missing_df

,Missing Count,Missing %
YEAR,0,0.0
DOY,0,0.0
T2M,0,0.0
T2M_MAX,0,0.0
T2M_MIN,0,0.0
T2M_RANGE,0,0.0
PRECTOTCORR,0,0.0
RH2M,0,0.0
WS2M,0,0.0
WS2M_MAX,0,0.0


- there are no null values, which makes our data interpretations and insights we draw from it more accurate since we dont miss anything

## 6. Outlier Detection

In [8]:
cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]

# Calculate z-scores
z_scores = np.abs(zscore(df[cols], nan_policy='omit'))

# Create named DataFrame for easier access
zscore_df = pd.DataFrame(
    z_scores, 
    columns=cols, 
    index=df.index
)

# FLAG the outliers (create a new column)
df['is_outlier'] = (zscore_df > 3).any(axis=1)  # ← THIS IS THE FLAG

# Report the counts
outliers = (z_scores > 3)
outlier_rows = df['is_outlier'].sum()
print("Outlier rows:", outlier_rows)
print(f"Percentage: {df['is_outlier'].mean()*100:.2f}%")

Outlier rows: 225
Percentage: 5.48%


- We have 225 rows with outlier values at one or more column, so now we check if they are erroneous values or seasonal extremities to make a decision whether to keep them or remove them.

### Check outliers by column

In [9]:
# 2. Check outliers by column
print_outliers(zscore_df, cols)

T2M: 10 outliers
T2M_MAX: 1 outliers
T2M_MIN: 68 outliers
PRECTOTCORR: 75 outliers
RH2M: 128 outliers
WS2M: 5 outliers
WS2M_MAX: 10 outliers


### Evaluating outliers

In [8]:
# 1. Physical possibility checks
impossible = df[
    (df["T2M_MAX"] < df["T2M_MIN"]) |  # Max < Min
    (df["RH2M"] > 100) |                # RH > 100%
    (df["RH2M"] < 0) |                  # RH < 0%
    (df["PRECTOTCORR"] < 0)             # Negative rain
]
print(f"Physically impossible rows: {len(impossible)}")

Physically impossible rows: 0


- We have no physically impossible rows, all values are within plausible ranges.

In [9]:
# 2. Check if outliers are seasonal
rain_outliers = df[zscore_df["PRECTOTCORR"] > 3] 
print("Precipitation outliers by month:")
print(rain_outliers["Month"].value_counts().sort_index())

Precipitation outliers by month:
Month
2      1
3      5
4      3
5      4
6     17
7     11
8      4
9     15
10    11
11     3
12     1
Name: count, dtype: int64


**Total daily precipitation (PRECTOTCORR)**

**Decision**: We will keep the precipitation outliers.

**Reason**

| Month | Outlier Count | Nigeria's Climate Zone |
|-------|---------------|------------------------|
| June | 17 | Peak of **Wet Season** (South & Middle Belt) |
| September | 15 | Second peak / End of Wet Season |
| July | 11 | Core Wet Season (August "little dry spell" explains slightly lower) |
| October | 11 | End of Wet Season / Start of Dry Season transition |


> Analysis of daily precipitation outliers (values exceeding the 95th percentile) reveals a strongly seasonal pattern. Extreme rainfall events occur exclusively during the wet season months (April–October), with peak frequencies in **June (17 events)** and **September (15 events)**. No extreme events were recorded during the dry season (November–March). This pattern is consistent with known West African monsoon dynamics and the bimodal rainfall distribution observed in parts of southern Nigeria. Therefore, these values are **not data anomalies but climatically normal extreme events** associated with deep convective storms during the monsoon period.

In [10]:
min_temp_outliers = df[zscore_df["T2M_MIN"] > 3]
print("Minimun temprature outliers by month:")
print(min_temp_outliers["Month"].value_counts().sort_index())

temp_outliers = df[zscore_df["T2M"] > 3]
print("Temprature outliers by month:")
print(temp_outliers["Month"].value_counts().sort_index())

max_temp_outliers = df[zscore_df["T2M_MAX"] > 3]
print("Maximum Temprature outliers by month:")
print(temp_outliers["Month"].value_counts().sort_index())

Minimun temprature outliers by month:
Month
1     41
2      1
12    26
Name: count, dtype: int64
Temprature outliers by month:
Month
1    10
Name: count, dtype: int64
Maximum Temprature outliers by month:
Month
1    10
Name: count, dtype: int64


**Decision**: We will keep the temperature outliers.

**Minimum Temperature Outliers**
**Distribution:** 
| Month | Outlier Count | Direction |
|-------|---------------|-----------|
| January | 41 | **Below normal** (colder nights) |
| February | 1 | Below normal |
| December | 26 | Below normal |

**Reasons:**

| Factor | Explanation |
|--------|-------------|
| **Harmattan season** | Dry, dusty winds from the Sahara (November–February) cause nighttime temperatures to drop sharply |
| **Clear skies** | No cloud cover at night allows rapid radiative cooling |
| **Low humidity** | Dry air cools faster than humid air |
| **December–January peak** | Coldest months in Nigeria (especially northern regions) |

**Mean Temperature Outliers**

**Distribution:** January(10)

**Reasons:**
- January combines **cool nights** (from Harmattan) with **mild days** (not extremely hot)
- This pulls the *daily mean* down enough to register as a statistical outlier
- December nights are cool, but December days are still warm → mean stays closer to normal


**Maximum Temperature Outliers**

**Distribution:** January(10)

**Reasons:**

| Factor | Explanation |
|--------|-------------|
| **Harmattan haze** | Thick dust reduces solar radiation reaching the ground |
| **Cloud cover** | Some January days may have more cloud cover |
| **Cold air advection** | Dry, cool air from the Sahara suppressing daytime heating |


> **All temperature outliers occur during the Harmattan season (December–February), with January being the peak month. These are not data errors but climatically normal cold spells associated with dry, dusty air from the Sahara. Minimum temperature outliers (41 events in January) are the most common, indicating that nighttime cooling is the dominant cold-season signal in this dataset. Daytime maximum temperature outliers are rarer (10 events) and occur only in January when thick Harmattan haze significantly reduces solar heating.**


In [11]:
relative_humidity_outliers = df[zscore_df["RH2M"] > 3]
print("Relative humidity outliers by month:")
print(relative_humidity_outliers["Month"].value_counts().sort_index())

Relative humidity outliers by month:
Month
1     52
2     17
11     2
12    57
Name: count, dtype: int64


**Decision**: We will keep the humidity outliers.

**Relative Humidity Outliers (RH2M)**
**Distribution:** December(57), January(52), February(17), November (2)

**Reason**:
> Low relative humidity outliers are strongly seasonal, occurring exclusively during the Harmattan season (November–February), with peak frequency in December (57 events) and January (52 events). These correspond precisely with the minimum temperature outliers identified earlier, confirming that the Harmattan period is characterized by **cool, dry, dusty air** from the Sahara. No low-humidity outliers occur during the wet season (March–October), when maritime monsoon air maintains consistently high humidity. These patterns are climatically normal for Nigeria and represent seasonal variation, not data errors.

In [12]:
mean_daily_wind_speed_outliers = df[zscore_df["WS2M"] > 3]
print("Mean daily wind speed outliers outliers by month:")
print(mean_daily_wind_speed_outliers["Month"].value_counts().sort_index())

max_daily_wind_speed_outliers = df[zscore_df["WS2M_MAX"] > 3]
print("Max daily wind speed outliers outliers by month:")
print(max_daily_wind_speed_outliers["Month"].value_counts().sort_index())

Mean daily wind speed outliers outliers by month:
Month
8    4
9    1
Name: count, dtype: int64
Max daily wind speed outliers outliers by month:
Month
8    8
9    2
Name: count, dtype: int64


**Decision**: We will keep the wind speed outliers.

**Mean Daily Wind Speed Outliers (WS2M)**
**Distribution:** August (4), September (1)


**Maximum Daily Wind Speed Outliers (WS2M_MAX)**
**Distribution:** August (8), September (2)

**Reason:**  
 
> High wind outliers occur exclusively in August and September, with maximum wind outliers (8 events) more frequent than mean wind outliers (4 events). This indicates **gusty conditions from squall lines and intense thunderstorms** rather than persistently strong winds. Unlike temperature and humidity outliers (which peak during the Harmattan dry season), wind outliers are a **late wet season phenomenon** associated with African Easterly Waves and the August "little dry spell." No wind outliers occur during the Harmattan months (November–February), despite popular perception of Harmattan as "windy" — Harmattan winds are steady and persistent but not statistically extreme.

## 7. Export Clean Data

In [13]:
df.to_csv(f"../data/nigeria_clean.csv", index=False)